In [2]:
from utils import *
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader

In [3]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using mps device


## Linear Regression

Simple population-level linear regression model. The country is not considered.

In [ ]:
X, Y, C = load_parkinsons_data()
model = LinearRegression().fit(X, Y)
y_pred = model.predict(X)
print("Loss:\n", np.mean((y_pred - Y)**2))
print("RMSD:\n", np.sqrt(np.mean((y_pred - Y)**2)))

Loss:
 45.53098
RMSD:
 6.747665


Next, we try learning a separate linear regression model for each country. Notice the loss is much better.

In [ ]:
X, Y, C = load_parkinsons_data()
model = ContextLinearRegression().fit(X, Y, C)
y_pred = model.predict(X, C)
print("Loss:\n", np.mean((y_pred - Y)**2))
print("RMSD:\n", np.sqrt(np.mean((y_pred - Y)**2)))

Loss:
 23.753902446756058
RMSD:
 4.87379753854795


## Multilayer Perceptron
The MLP without country as input performs about equally well as learning a separate linear regression model for each country.

In [4]:
N_FEATURES = 16

model = NeuralNetwork(dim_in=N_FEATURES, dim_out=1, dim_hidden=300, n_hidden=1).to(device)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

full_dataset = ParkinsonsDataset()

training_data, test_data = torch.utils.data.random_split(full_dataset, [0.8, 0.2])
train_dataloader = DataLoader(training_data, batch_size=50)
test_dataloader = DataLoader(test_data, batch_size=50)

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer, device)
    test_loss = test(test_dataloader, model, loss_fn, device)
print("Done!")

print("Loss:", test_loss.item())
print("RMSD:", np.sqrt(test_loss.item()))

Epoch 1
-------------------------------


TypeError: Cannot convert a MPS Tensor to float64 dtype as the MPS framework doesn't support float64. Please use float32 instead.

## Context-Sensitive Network
The task is one-hot-encoded and appended to the input.

In [ ]:
model = NeuralNetwork(dim_in=203, dim_out=1, dim_hidden=300, n_hidden=3).to(device)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

full_dataset = ContextSensitiveHeightDataset()

training_data, test_data = torch.utils.data.random_split(full_dataset, [0.8, 0.2])
train_dataloader = DataLoader(training_data, batch_size=50)
test_dataloader = DataLoader(test_data, batch_size=50)

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer, device)
    test_loss = test(test_dataloader, model, loss_fn, device)
print("Done!")

print("Loss:", test_loss.item())
print("RMSD:", np.sqrt(test_loss.item()))

Epoch 1
-------------------------------
loss: 331.746307  [   50/168000]
loss: 40.781631  [ 5050/168000]
loss: 25.461664  [10050/168000]
loss: 40.032555  [15050/168000]
loss: 43.459724  [20050/168000]
loss: 34.916988  [25050/168000]
loss: 31.376133  [30050/168000]
loss: 31.249811  [35050/168000]
loss: 26.349007  [40050/168000]
loss: 29.113007  [45050/168000]
loss: 24.952339  [50050/168000]
loss: 22.770981  [55050/168000]
loss: 29.075806  [60050/168000]
loss: 23.920170  [65050/168000]
loss: 28.978153  [70050/168000]
loss: 23.888945  [75050/168000]
loss: 15.056357  [80050/168000]
loss: 23.623674  [85050/168000]
loss: 15.057790  [90050/168000]
loss: 17.082714  [95050/168000]
loss: 19.912663  [100050/168000]
loss: 18.221647  [105050/168000]
loss: 22.354872  [110050/168000]
loss: 18.768999  [115050/168000]
loss: 21.283648  [120050/168000]
loss: 18.665510  [125050/168000]
loss: 17.913023  [130050/168000]
loss: 19.501816  [135050/168000]
loss: 17.890760  [140050/168000]
loss: 18.838717  [1450

## Learned-Context Neural Network
Instead of one-hot encoding, the task is represented as a learned context parameter and appended to the input.

In [ ]:
model = LearnedContextNN(dim_in=3, dim_out=1, dim_hidden=300, n_hidden=3, dim_context=30, n_context=200).to(device)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

full_dataset = ContextSensitiveHeightDataset()

training_data, test_data = torch.utils.data.random_split(full_dataset, [0.8, 0.2])
train_dataloader = DataLoader(training_data, batch_size=50)
test_dataloader = DataLoader(test_data, batch_size=50)

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer, device)
    test_loss = test(test_dataloader, model, loss_fn, device)
print("Done!")

print("Loss:", test_loss.item())
print("RMSD:", np.sqrt(test_loss.item()))

Epoch 1
-------------------------------
loss: 361.046021  [   50/168000]
loss: 30.001230  [ 5050/168000]
loss: 36.133129  [10050/168000]
loss: 25.951054  [15050/168000]
loss: 24.040258  [20050/168000]
loss: 24.041294  [25050/168000]
loss: 32.219769  [30050/168000]
loss: 20.430801  [35050/168000]
loss: 23.104195  [40050/168000]
loss: 13.644760  [45050/168000]
loss: 15.912150  [50050/168000]
loss: 9.618295  [55050/168000]
loss: 8.372338  [60050/168000]
loss: 10.065187  [65050/168000]
loss: 9.260690  [70050/168000]
loss: 10.783900  [75050/168000]
loss: 4.434227  [80050/168000]
loss: 2.501760  [85050/168000]
loss: 3.199759  [90050/168000]
loss: 5.092778  [95050/168000]
loss: 4.658932  [100050/168000]
loss: 2.494370  [105050/168000]
loss: 8.040448  [110050/168000]
loss: 3.009257  [115050/168000]
loss: 1.914656  [120050/168000]
loss: 2.254064  [125050/168000]
loss: 3.730703  [130050/168000]
loss: 1.916394  [135050/168000]
loss: 2.196719  [140050/168000]
loss: 2.347542  [145050/168000]
loss: 